In [ ]:
# ================================================================
# CELL 1: Mount Google Drive
# ================================================================

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ================================================================
# CELL 2: Extract Dataset and Install
# ================================================================

import zipfile
import os

zip_path = "/content/drive/MyDrive/Research/dataset/dataset_500/dataset.zip"
extract_path = "/content/data/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

DATASET_PATH = "/content/data/dataset"

for root, dirs, files in os.walk(extract_path):
    for d in dirs:
        subfolder = os.path.join(root, d)
        if any(f.endswith('.tif') for f in os.listdir(subfolder)):
            DATASET_PATH = root
            break

print(f"Dataset path: {DATASET_PATH}")
print(f"Classes: {sorted(os.listdir(DATASET_PATH))}")

!pip install rasterio opencv-python cma

In [ ]:
# ================================================================
# CELL 3: Imports
# ================================================================

import os
import gc
import json
import pickle
import numpy as np
import rasterio
import cv2
import cma
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

tf.get_logger().setLevel('ERROR')

SAVE_DIR = "/content/drive/MyDrive/Research/cmaes_results/rgb_3band_5000"
MASTER_RESULTS_FILE = "/content/drive/MyDrive/Research/all_experiment_results/master_results.json"
CHECKPOINT_PATH = os.path.join(SAVE_DIR, "cma_checkpoint.pkl")
os.makedirs(SAVE_DIR, exist_ok=True)

print("All imports done!")

In [ ]:
# ================================================================
# CELL 4: Load Dataset (3-band RGB, 5000 images)
# ================================================================

IMG_SIZE = 64

def load_rgb_image(tif_path):
    with rasterio.open(tif_path) as src:
        img = src.read([4, 3, 2]).astype(np.float32)
    img = np.transpose(img, (1, 2, 0))
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    img = img / 10000.0
    return img

X = []
y = []

class_names = sorted(os.listdir(DATASET_PATH))
class_to_idx = {name: idx for idx, name in enumerate(class_names)}

print("Class mapping:")
for k, v in class_to_idx.items():
    print(f"  {k} → {v}")

for class_name in class_names:
    class_folder = os.path.join(DATASET_PATH, class_name)
    if not os.path.isdir(class_folder):
        continue
    label = class_to_idx[class_name]
    for file in os.listdir(class_folder):
        if file.endswith(".tif"):
            img_path = os.path.join(class_folder, file)
            img = load_rgb_image(img_path)
            X.append(img)
            y.append(label)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

print(f"\nDataset: X={X.shape}, y={y.shape}")

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

del X, y, X_temp, y_temp
gc.collect()

# Convert to float16 to save RAM
X_train = X_train.astype(np.float16)
X_val = X_val.astype(np.float16)
X_test = X_test.astype(np.float16)

print(f"Train: {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test: {X_test.shape}")

In [ ]:
# ================================================================
# CELL 5: Model Building Function
# ================================================================

activation_map = {
    0: "relu",
    1: "tanh"
}

def build_cnn_model(
    input_shape,
    num_classes,
    num_conv_layers,
    filters_list,
    kernel_size,
    activation,
    dropout_rate
):
    model = models.Sequential()
    model.add(layers.Input(shape=input_shape))

    for i in range(num_conv_layers):
        model.add(
            layers.Conv2D(
                filters=filters_list[i],
                kernel_size=(kernel_size, kernel_size),
                padding="same",
                activation=activation
            )
        )
        model.add(layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation=activation))
    model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(num_classes, activation="softmax"))

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

print("Model building function ready!")

In [ ]:
# ================================================================
# CELL 6: CMA-ES Search Space and Decode Function
# ================================================================

BOUNDS = {
    "num_conv_layers": (1, 3),
    "filters_1": (16, 64),
    "filters_2": (32, 128),
    "filters_3": (64, 256),
    "kernel_size": (3, 5),
    "activation_id": (0, 1),
    "dropout_rate": (0.0, 0.5)
}

def decode_architecture(x):
    num_conv_layers = int(np.clip(round(x[0]), *BOUNDS["num_conv_layers"]))
    filters_1 = int(np.clip(round(x[1]), *BOUNDS["filters_1"]))
    filters_2 = int(np.clip(round(x[2]), *BOUNDS["filters_2"]))
    filters_3 = int(np.clip(round(x[3]), *BOUNDS["filters_3"]))
    kernel_size = int(np.clip(round(x[4]), *BOUNDS["kernel_size"]))
    activation_id = int(np.clip(round(x[5]), *BOUNDS["activation_id"]))
    activation = activation_map[activation_id]
    dropout_rate = float(np.clip(x[6], *BOUNDS["dropout_rate"]))

    filters_list = [filters_1]
    if num_conv_layers > 1:
        filters_list.append(filters_2)
    if num_conv_layers > 2:
        filters_list.append(filters_3)

    return {
        "num_conv_layers": num_conv_layers,
        "filters_list": filters_list,
        "kernel_size": kernel_size,
        "activation": activation,
        "dropout_rate": dropout_rate
    }

print("Search space ready!")

In [ ]:
# ================================================================
# CELL 7: CMA-ES Search WITH CHECKPOINT/RESUME
#
# After it finishes 5 generations:
#   1. It will print "⚠️  NOW DO THIS:"
#   2. Go to Runtime → Restart runtime
#   3. Run ALL cells from Cell 1 to Cell 7 again
#   4. It will auto-resume from where it stopped
#   5. Repeat until all 10 generations are done
# ================================================================

# --- SETTINGS ---
TOTAL_GENERATIONS = 10
GENS_PER_CHUNK = 5
POPSIZE = 20
EPOCHS_PER_CANDIDATE = 10

x0 = [3, 32, 64, 128, 3, 0, 0.5]
sigma = 0.5

# --- LOAD OR CREATE CMA-ES STATE ---
if os.path.exists(CHECKPOINT_PATH):
    print("📂 Loading checkpoint...")
    with open(CHECKPOINT_PATH, 'rb') as f:
        checkpoint = pickle.load(f)

    es = checkpoint['es']
    best_val_accuracy = checkpoint['best_val_accuracy']
    best_architecture = checkpoint['best_architecture']
    completed_generations = checkpoint['completed_generations']
    all_search_results = checkpoint['all_search_results']
    print(f"✅ Resuming from generation {completed_generations + 1}")
    print(f"🏆 Best so far: {best_val_accuracy:.4f}")
else:
    print("🆕 Starting fresh CMA-ES search (3-band RGB, 5000 images)...")
    es = cma.CMAEvolutionStrategy(x0, sigma, {
        "popsize": POPSIZE,
        "verb_log": 0
    })
    best_val_accuracy = 0.0
    best_architecture = None
    completed_generations = 0
    all_search_results = []

# --- FITNESS FUNCTION ---
def fitness_function(x):
    global best_val_accuracy, best_architecture, all_search_results

    tf.keras.backend.clear_session()
    gc.collect()

    arch = decode_architecture(x)

    print(f"  Layers: {arch['num_conv_layers']}, Filters: {arch['filters_list']}, "
          f"Kernel: {arch['kernel_size']}, Act: {arch['activation']}, "
          f"Drop: {arch['dropout_rate']:.2f}")

    model = build_cnn_model(
        input_shape=(64, 64, 3),
        num_classes=10,
        num_conv_layers=arch["num_conv_layers"],
        filters_list=arch["filters_list"],
        kernel_size=arch["kernel_size"],
        activation=arch["activation"],
        dropout_rate=arch["dropout_rate"]
    )

    model.fit(
        X_train, y_train,
        epochs=EPOCHS_PER_CANDIDATE,
        batch_size=32,
        validation_data=(X_val, y_val),
        verbose=0
    )

    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    print(f"  Val Accuracy: {val_acc:.4f}")

    if val_acc > best_val_accuracy:
        best_val_accuracy = float(val_acc)
        best_architecture = {
            'num_conv_layers': arch['num_conv_layers'],
            'filters_list': arch['filters_list'],
            'kernel_size': arch['kernel_size'],
            'activation': arch['activation'],
            'dropout_rate': arch['dropout_rate']
        }
        try:
            model.save(os.path.join(SAVE_DIR, 'best_model_search.h5'))
            print(f"  💾 New best! Saved.")
        except Exception as e:
            print(f"  Could not save: {e}")

    fitness = 1.0 - val_acc

    all_search_results.append({
        'generation': completed_generations + 1,
        'architecture': arch,
        'val_acc': float(val_acc)
    })

    del model
    gc.collect()

    return fitness

# --- RUN GENERATIONS ---
remaining = TOTAL_GENERATIONS - completed_generations
gens_to_run = min(GENS_PER_CHUNK, remaining)

if gens_to_run <= 0:
    print(f"\n🏁 ALL {TOTAL_GENERATIONS} GENERATIONS ALREADY COMPLETED!")
    print(f"🏆 Best Validation Accuracy: {best_val_accuracy:.4f}")
    if best_architecture:
        for k, v in best_architecture.items():
            print(f"   {k}: {v}")
else:
    print(f"\n🚀 Running generations {completed_generations+1} to {completed_generations+gens_to_run} out of {TOTAL_GENERATIONS}")

    for g in range(gens_to_run):
        gen_num = completed_generations + g + 1
        print(f"\n{'='*50}")
        print(f"  Generation {gen_num}/{TOTAL_GENERATIONS}")
        print(f"{'='*50}")

        solutions = es.ask()
        fitnesses = []

        for i, x in enumerate(solutions):
            print(f"\n--- Candidate {i+1}/{POPSIZE} (Gen {gen_num}) ---")
            f = fitness_function(x)
            fitnesses.append(f)

        es.tell(solutions, fitnesses)
        es.disp()
        print(f"\n🏆 Best after Gen {gen_num}: {best_val_accuracy:.4f}")

    completed_generations += gens_to_run

    # --- SAVE CHECKPOINT ---
    checkpoint = {
        'es': es,
        'best_val_accuracy': best_val_accuracy,
        'best_architecture': best_architecture,
        'completed_generations': completed_generations,
        'all_search_results': all_search_results
    }
    with open(CHECKPOINT_PATH, 'wb') as f:
        pickle.dump(checkpoint, f)

    print(f"\n{'='*50}")
    print(f"💾 CHECKPOINT SAVED!")
    print(f"   Completed: {completed_generations}/{TOTAL_GENERATIONS} generations")
    print(f"   Best accuracy: {best_val_accuracy:.4f}")
    if best_architecture:
        for k, v in best_architecture.items():
            print(f"   {k}: {v}")
    print(f"{'='*50}")

    if completed_generations < TOTAL_GENERATIONS:
        print(f"\n⚠️  NOW DO THIS:")
        print(f"   1. Go to Runtime → Restart runtime")
        print(f"   2. Run ALL cells again (Cell 1 through Cell 7)")
        print(f"   3. It will auto-resume from generation {completed_generations+1}")
        print(f"   4. Repeat until all {TOTAL_GENERATIONS} generations are done")
    else:
        print(f"\n🏁 ALL {TOTAL_GENERATIONS} GENERATIONS COMPLETE!")

# Save search results
search_data = {
    'config': '3-band RGB, 5000 images',
    'generations': TOTAL_GENERATIONS,
    'population': POPSIZE,
    'epochs_per_candidate': EPOCHS_PER_CANDIDATE,
    'completed_generations': completed_generations,
    'best_val_accuracy': best_val_accuracy,
    'best_architecture': best_architecture,
    'all_results': all_search_results
}
with open(os.path.join(SAVE_DIR, 'search_results.json'), 'w') as f:
    json.dump(search_data, f, indent=2)

In [ ]:
# ================================================================
# CELL 8: Retrain Best CMA-ES Architecture (100 epochs)
# ⚠️ ONLY RUN THIS AFTER ALL 10 GENERATIONS ARE COMPLETE
# ================================================================

print("=" * 60)
print("  RETRAINING BEST CMA-ES ARCHITECTURE (100 epochs)")
print("  Config: 3-band RGB, 5000 images")
print("=" * 60)

tf.keras.backend.clear_session()
gc.collect()

# Load best architecture from search results if needed
if best_architecture is None:
    with open(os.path.join(SAVE_DIR, 'search_results.json'), 'r') as f:
        search_data = json.load(f)
    best_architecture = search_data['best_architecture']
    best_val_accuracy = search_data['best_val_accuracy']

print(f"Architecture: {best_architecture}")

cmaes_model = build_cnn_model(
    input_shape=(64, 64, 3),
    num_classes=10,
    num_conv_layers=best_architecture['num_conv_layers'],
    filters_list=best_architecture['filters_list'],
    kernel_size=best_architecture['kernel_size'],
    activation=best_architecture['activation'],
    dropout_rate=best_architecture['dropout_rate']
)

cmaes_model.summary()

my_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=5, min_lr=1e-6, verbose=1
    )
]

history_cmaes = cmaes_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=my_callbacks,
    verbose=1
)

cmaes_val_loss, cmaes_val_acc = cmaes_model.evaluate(X_val, y_val, verbose=0)
cmaes_test_loss, cmaes_test_acc = cmaes_model.evaluate(X_test, y_test, verbose=0)

print(f"\n📊 CMA-ES Validation Accuracy: {cmaes_val_acc:.4f}")
print(f"📊 CMA-ES Test Accuracy:       {cmaes_test_acc:.4f}")

y_pred_cmaes = np.argmax(cmaes_model.predict(X_test), axis=1)

cmaes_model.save(os.path.join(SAVE_DIR, 'cmaes_retrained_model.h5'))

cmaes_history = {
    'accuracy': [float(x) for x in history_cmaes.history['accuracy']],
    'val_accuracy': [float(x) for x in history_cmaes.history['val_accuracy']],
    'loss': [float(x) for x in history_cmaes.history['loss']],
    'val_loss': [float(x) for x in history_cmaes.history['val_loss']],
}
with open(os.path.join(SAVE_DIR, 'cmaes_training_history.json'), 'w') as f:
    json.dump(cmaes_history, f)

cmaes_report = classification_report(
    y_test, y_pred_cmaes,
    target_names=class_names,
    digits=4,
    output_dict=True
)

del cmaes_model
gc.collect()

print("✅ CMA-ES retraining complete!")

In [ ]:
# ================================================================
# CELL 9: Load Baseline Results and Compare
# ================================================================

print("=" * 60)
print("  COMPARISON: BASELINE vs CMA-ES")
print("  Config: 3-band RGB, 5000 images")
print("=" * 60)

with open(MASTER_RESULTS_FILE, 'r') as f:
    master_data = json.load(f)

baseline_entry = next(
    (r for r in master_data['experiments']
     if r['num_bands'] == 3 and r['num_images'] == '5000'),
    None
)

if baseline_entry is None:
    print("❌ Baseline result for 3-band 5000 not found in master table!")
    print("   Run the baseline notebook first.")
else:
    baseline_val_acc = baseline_entry['val_accuracy']
    baseline_test_acc = baseline_entry['test_accuracy']
    baseline_f1 = baseline_entry['macro_f1']

    print("\nCMA-ES CNN Classification Report:")
    print(classification_report(y_test, y_pred_cmaes, target_names=class_names, digits=4))

    print(f"\n{'='*65}")
    print(f"{'Metric':<25} {'Baseline':>12} {'CMA-ES':>12} {'Difference':>12}")
    print("-" * 65)
    print(f"{'Validation Accuracy':<25} {baseline_val_acc:>12.4f} {cmaes_val_acc:>12.4f} {cmaes_val_acc - baseline_val_acc:>+12.4f}")
    print(f"{'Test Accuracy':<25} {baseline_test_acc:>12.4f} {cmaes_test_acc:>12.4f} {cmaes_test_acc - baseline_test_acc:>+12.4f}")
    print(f"{'Macro F1-Score':<25} {baseline_f1:>12.4f} {cmaes_report['macro avg']['f1-score']:>12.4f} {cmaes_report['macro avg']['f1-score'] - baseline_f1:>+12.4f}")
    print("-" * 65)

    if 'per_class_f1' in baseline_entry:
        print(f"\n{'Class':<20} {'Baseline F1':>12} {'CMA-ES F1':>12} {'Diff':>12}")
        print("-" * 56)
        for cls in class_names:
            b = baseline_entry['per_class_f1'].get(cls, 0)
            c = cmaes_report[cls]['f1-score']
            print(f"{cls:<20} {b:>12.4f} {c:>12.4f} {c - b:>+12.4f}")

    print(f"\n  Baseline architecture: 3 layers, [32,64,128], kernel 3, relu, dropout 0.5")
    print(f"  CMA-ES architecture:  {best_architecture['num_conv_layers']} layers, "
          f"{best_architecture['filters_list']}, kernel {best_architecture['kernel_size']}, "
          f"{best_architecture['activation']}, dropout {best_architecture['dropout_rate']:.2f}")

In [ ]:
# ================================================================
# CELL 10: Save to Master Results Table
# ================================================================

master_data['cmaes_experiments']['3-band_5000'] = {
    'val_accuracy': float(cmaes_val_acc),
    'test_accuracy': float(cmaes_test_acc),
    'macro_f1': float(cmaes_report['macro avg']['f1-score']),
    'architecture': best_architecture,
    'search_config': {
        'generations': TOTAL_GENERATIONS,
        'population': POPSIZE,
        'epochs_per_candidate': EPOCHS_PER_CANDIDATE,
        'total_evaluated': len(all_search_results)
    }
}

with open(MASTER_RESULTS_FILE, 'w') as f:
    json.dump(master_data, f, indent=2)

print(f"💾 CMA-ES results saved to master table!")

print(f"\n{'='*70}")
print("  MASTER TABLE STATUS")
print(f"{'='*70}")
print(f"{'Configuration':<25} {'Baseline Test':>14} {'CMA-ES Test':>12} {'Status':>10}")
print("-" * 70)

cmaes_exps = master_data.get('cmaes_experiments', {})
for r in master_data.get('experiments', []):
    key = f"{r['num_bands']}-band_{r['num_images']}"
    cmaes = cmaes_exps.get(key, {})
    c_test = cmaes.get('test_accuracy')

    c_str = f"{c_test:.4f}" if c_test else "---"
    status = "✅ Done" if c_test else "⏳ Pending"
    label = f"{r['num_bands']}-band, {r['num_images']} imgs"

    print(f"{label:<25} {r['test_accuracy']:>14.4f} {c_str:>12} {status:>10}")

print("-" * 70)

In [ ]:
# ================================================================
# CELL 11: Generate Charts
# ================================================================

# ---- Training curves ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(cmaes_history['accuracy'], label='CMA-ES Train', linestyle='-', color='red')
axes[0].plot(cmaes_history['val_accuracy'], label='CMA-ES Val', linestyle='--', color='red')
axes[0].set_title('CMA-ES Model Accuracy (3-band, 5000 imgs)', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(cmaes_history['loss'], label='CMA-ES Train', linestyle='-', color='red')
axes[1].plot(cmaes_history['val_loss'], label='CMA-ES Val', linestyle='--', color='red')
axes[1].set_title('CMA-ES Model Loss (3-band, 5000 imgs)', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=200, bbox_inches='tight')
plt.show()

# ---- Confusion matrix ----
cm_cmaes = confusion_matrix(y_test, y_pred_cmaes)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm_cmaes, interpolation='nearest', cmap=plt.cm.Blues)
ax.set_title('CMA-ES CNN Confusion Matrix (3-band, 5000 imgs)', fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.046)

thresh = cm_cmaes.max() / 2.0
for i in range(cm_cmaes.shape[0]):
    for j in range(cm_cmaes.shape[1]):
        ax.text(j, i, format(cm_cmaes[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm_cmaes[i, j] > thresh else "black", fontsize=8)

ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(class_names, fontsize=8)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'), dpi=200, bbox_inches='tight')
plt.show()

# ---- Per-class F1 comparison ----
if baseline_entry and 'per_class_f1' in baseline_entry:
    fig, ax = plt.subplots(figsize=(12, 5))

    b_f1 = [baseline_entry['per_class_f1'].get(cls, 0) for cls in class_names]
    c_f1 = [cmaes_report[cls]['f1-score'] for cls in class_names]
    x = np.arange(len(class_names))
    width = 0.35

    ax.bar(x - width/2, b_f1, width, label='Baseline CNN', color='steelblue')
    ax.bar(x + width/2, c_f1, width, label='CMA-ES CNN', color='indianred')
    ax.set_xlabel('Land Use Class')
    ax.set_ylabel('F1-Score')
    ax.set_title('Per-Class F1: Baseline vs CMA-ES (3-band, 5000 imgs)', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=8)
    ax.legend()
    ax.set_ylim(0, 1.1)
    ax.grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'per_class_f1.png'), dpi=200, bbox_inches='tight')
    plt.show()

# ---- Search progress ----
fig, ax = plt.subplots(figsize=(10, 5))

gen_numbers = []
gen_best = []
gen_avg = []

for gen in range(1, completed_generations + 1):
    gen_results = [r for r in all_search_results if r['generation'] == gen]
    if gen_results:
        gen_numbers.append(gen)
        gen_best.append(max(r['val_acc'] for r in gen_results))
        gen_avg.append(np.mean([r['val_acc'] for r in gen_results]))

ax.plot(gen_numbers, gen_best, 'ro-', label='Best in Generation', linewidth=2, markersize=8)
ax.plot(gen_numbers, gen_avg, 'bs--', label='Average in Generation', linewidth=1.5, markersize=6)
ax.set_xlabel('Generation')
ax.set_ylabel('Validation Accuracy')
ax.set_title('CMA-ES Search Progress (3-band, 5000 imgs)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(gen_numbers)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'search_progress.png'), dpi=200, bbox_inches='tight')
plt.show()

print(f"\n💾 All charts saved to {SAVE_DIR}")

In [ ]:
# ================================================================
# CELL 12: Final Summary
# ================================================================

b_test = baseline_entry['test_accuracy'] if baseline_entry else 0
b_val = baseline_entry['val_accuracy'] if baseline_entry else 0

print("=" * 60)
print("  COMPLETE RESULTS SUMMARY")
print("  Experiment: CMA-ES, 3-band RGB, 5000 images")
print("=" * 60)

print(f"""
DATASET:
  Total images: 5,000 (500 per class)
  Training: {X_train.shape[0]}
  Validation: {X_val.shape[0]}
  Test: {X_test.shape[0]}
  Input shape: (64, 64, 3)
  Classes: 10

CMA-ES SEARCH:
  Generations: {TOTAL_GENERATIONS}
  Population: {POPSIZE}
  Epochs per candidate: {EPOCHS_PER_CANDIDATE}
  Architectures evaluated: {len(all_search_results)}
  Best architecture found:
    Layers: {best_architecture['num_conv_layers']}
    Filters: {best_architecture['filters_list']}
    Kernel: {best_architecture['kernel_size']}
    Activation: {best_architecture['activation']}
    Dropout: {best_architecture['dropout_rate']}

BASELINE CNN (from baseline notebook):
  Architecture: 3 layers, [32,64,128], kernel 3, relu, dropout=0.5
  Validation Accuracy: {b_val:.4f}
  Test Accuracy: {b_test:.4f}

CMA-ES CNN (retrained 100 epochs):
  Validation Accuracy: {cmaes_val_acc:.4f}
  Test Accuracy: {cmaes_test_acc:.4f}

IMPROVEMENT:
  Validation: {cmaes_val_acc - b_val:+.4f} ({(cmaes_val_acc - b_val)*100:+.2f}%)
  Test:       {cmaes_test_acc - b_test:+.4f} ({(cmaes_test_acc - b_test)*100:+.2f}%)

FILES SAVED:
  {SAVE_DIR}/search_results.json
  {SAVE_DIR}/best_model_search.h5
  {SAVE_DIR}/cmaes_retrained_model.h5
  {SAVE_DIR}/cmaes_training_history.json
  {SAVE_DIR}/training_curves.png
  {SAVE_DIR}/confusion_matrix.png
  {SAVE_DIR}/per_class_f1.png
  {SAVE_DIR}/search_progress.png
  {MASTER_RESULTS_FILE} (updated)
""")

print("🏁 DONE! Next: run CMA-ES notebook for 13-band 5000 images.")